# PRO-cap Atlas BPNet locus viewer

Visualize strand-specific observed PRO-cap signal, fold-averaged BPNet predictions generated locally from downloaded model checkpoints, and frequency-reference DeepLIFT/SHAP logos at one locus.

This notebook is intentionally lighter than `procap_atlas_bpnet_locus_viewer.ipynb`: it does not run shuffled-reference diagnostics. DeepLIFT uses the same production-style genomic nucleotide-frequency null as `src/bpnet/attribute/attribute_bpnet.py`: one soft reference per input sequence whose A/C/G/T probabilities are the observed frequencies across the 2114 bp input.

In [ ]:
from pathlib import Path as _Path
import importlib.util as _importlib_util
import os as _os
import sys as _sys

_numpy_spec = _importlib_util.find_spec("numpy")
_numpy_origin = _Path(_numpy_spec.origin).resolve() if _numpy_spec else None
_environment_root = _Path(_sys.prefix).resolve()
print(f"Python: {_sys.executable}")
print(f"Environment: {_environment_root}")
print(f"NumPy candidate: {_numpy_origin}")
print(f"PYTHONPATH: {_os.environ.get('PYTHONPATH')!r}")
if _os.environ.get("PYTHONPATH"):
    raise RuntimeError("The PRO-cap Atlas kernel did not remove the Open OnDemand PYTHONPATH. Reinstall it with notebooks/install_uv_kernel.py and restart JupyterLab.")
_ondemand_paths = [path for path in _sys.path if path.startswith("/share/software/user/open/py-jupyterlab/")]
if _ondemand_paths:
    raise RuntimeError(f"Open OnDemand PYTHONPATH entries are affecting imports: {_ondemand_paths}. Reinstall the kernel with notebooks/install_uv_kernel.py.")
if _numpy_origin is None or not _numpy_origin.is_relative_to(_environment_root):
    raise RuntimeError("NumPy is not resolving from the selected uv environment. Reinstall the PRO-cap Atlas kernel with notebooks/install_uv_kernel.py.")

In [ ]:
from pathlib import Path
import gc
import gzip
import os
import shutil
import sys
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pybigtools
import seaborn as sns
import torch
import yaml
from bpnetlite.bpnet import CountWrapper, ProfileWrapper
from huggingface_hub import hf_hub_download
from pyfaidx import Fasta
from tangermeme.io import extract_loci
from tangermeme.plot import plot_logo
from tangermeme.predict import predict

for _parent in [Path.cwd(), *Path.cwd().parents]:
    if (_parent / "src").is_dir() and str(_parent) not in sys.path:
        sys.path.insert(0, str(_parent))
        break

from src.bpnet.attribute.deeplift import deep_lift_shap
from src.bpnet.attribute.locus_diagnostics import (
    as_numpy,
    genomic_offsets,
    reverse_complement_matrix,
    reverse_complement_tracks,
)

sns.set_style("whitegrid")

MODEL_REPO_ID = "adamyhe/procap-atlas"
TRACK_REPO_ID = "adamyhe/procap-atlas-tracks"
METADATA_REPO_ID = "adamyhe/procap-atlas-metadata"
REFERENCE_FASTA_URL = "https://www.encodeproject.org/files/GRCh38_no_alt_analysis_set_GCA_000001405.15/@@download/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta.gz"
IN_WINDOW = 2114
OUT_WINDOW = 1000

## Configuration

`POINT_REGION` centers the 2114 bp model input. `VIEW_REGION` controls the observed/predicted signal panel and must fit inside the model's 1000 bp output. `LOGO_REGION` controls the DeepLIFT logo crop and must fit inside the model input.

In [ ]:
EXP_ID = "ENCSR342WAR"
POINT_REGION = "chr2:181680717"
VIEW_REGION = "chr2:181680467-181681166"
LOGO_REGION = "chr2:181680467-181681167"
REVERSE_COMPLEMENT = False
N_FOLDS = 7
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = Path(os.environ.get("SCRATCH", ".cache")) / "procap_atlas_locus_viewer"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Work directory: {WORK_DIR}")

In [ ]:
def parse_point(region):
    chrom, position = region.replace(",", "").split(":", 1)
    return chrom, int(position) - 1


def parse_interval(region):
    chrom, interval = region.replace(",", "").split(":", 1)
    start, end = [int(value) for value in interval.split("-", 1)]
    if end < start:
        raise ValueError("Interval end must not precede its start")
    return chrom, start - 1, end


def download_first(repo_id, repo_type, filenames):
    last_error = None
    for filename in filenames:
        try:
            return Path(hf_hub_download(repo_id=repo_id, repo_type=repo_type, filename=filename))
        except Exception as error:
            last_error = error
    raise FileNotFoundError(f"Could not find any of {filenames} in {repo_id}") from last_error


def download_reference(work_dir=WORK_DIR):
    reference_dir = work_dir / "reference"
    reference_dir.mkdir(parents=True, exist_ok=True)
    fasta = reference_dir / "hg38.fa"
    compressed = reference_dir / "hg38.fa.gz"
    if not fasta.exists():
        if not compressed.exists():
            urllib.request.urlretrieve(REFERENCE_FASTA_URL, compressed)
        with gzip.open(compressed, "rb") as source, open(fasta, "wb") as target:
            shutil.copyfileobj(source, target)
    if not Path(str(fasta) + ".fai").exists():
        Fasta(str(fasta))
    return fasta


def download_model_paths(exp_id=EXP_ID, n_folds=N_FOLDS):
    patterns = [
        "{exp_id}/{exp_id}.fold{fold}.torch",
        "bpnet/{exp_id}/{exp_id}.fold{fold}.torch",
        "models/bpnet/{exp_id}/{exp_id}.fold{fold}.torch",
    ]
    return [
        download_first(
            MODEL_REPO_ID,
            "model",
            [pattern.format(exp_id=exp_id, fold=fold) for pattern in patterns],
        )
        for fold in range(n_folds)
    ]


def setup_experiment(exp_id=EXP_ID):
    config_path = download_first(
        METADATA_REPO_ID,
        "dataset",
        ["experiment_config.yaml", "configs/experiment_config.yaml"],
    )
    with open(config_path) as handle:
        config = yaml.safe_load(handle)["experiments"]
    if exp_id not in config:
        raise KeyError(f"{exp_id} is absent from experiment_config.yaml")
    return {
        "exp_id": exp_id,
        "config": config[exp_id],
        "fasta": download_reference(),
        "observed": {
            "plus": download_first(TRACK_REPO_ID, "dataset", [f"observed/{exp_id}_pl.bigWig"]),
            "minus": download_first(TRACK_REPO_ID, "dataset", [f"observed/{exp_id}_mn.bigWig"]),
        },
        "model_paths": download_model_paths(exp_id),
    }


def locus_input(resources):
    chrom, center = parse_point(POINT_REGION)
    loci = pd.DataFrame({"chrom": [chrom], "start": [center], "end": [center + 1]})
    X = extract_loci(
        loci,
        sequences=str(resources["fasta"]),
        in_window=IN_WINDOW,
        ignore=["N", "n"],
    )
    if len(X) != 1:
        raise ValueError("The requested locus could not be extracted")
    return chrom, center, X.float()


def nucleotide_frequency_references(X):
    frequencies = X.float().mean(dim=-1, keepdim=True)
    return frequencies.expand_as(X).unsqueeze(1).clone()

In [ ]:
resources = setup_experiment()
chrom, center, X = locus_input(resources)
logo_chrom, logo_start, logo_end = parse_interval(LOGO_REGION)
if logo_chrom != chrom:
    raise ValueError("POINT_REGION and LOGO_REGION must use the same chromosome")
logo_offsets = genomic_offsets(center, logo_start, logo_end, IN_WINDOW)
print(resources["config"].get("biosample", EXP_ID), X.shape, logo_offsets)

## Predictions and observed tracks

Predicted signal is generated from every selected fold checkpoint and averaged after converting profile logits to count-scale signal with `softmax(profile_logits) * exp(log_counts)`. Minus-strand tracks are plotted as negative values.

In [ ]:
def scaled_prediction(model, X):
    logits, log_counts = predict(model=model, X=X, batch_size=1, device=DEVICE)
    logits = as_numpy(logits).astype(np.float32)
    flat = logits.reshape(logits.shape[0], -1)
    probabilities = np.exp(flat - flat.max(axis=1, keepdims=True))
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    counts = np.exp(as_numpy(log_counts).reshape(logits.shape[0], -1))
    return (probabilities * counts).reshape(logits.shape)[0]


def ensemble_prediction(resources, X):
    predictions = []
    for fold, path in enumerate(resources["model_paths"][:N_FOLDS]):
        print(f"Predicting fold {fold + 1}/{N_FOLDS}: {path.name}")
        model = torch.load(path, map_location="cpu", weights_only=False).eval()
        predictions.append(scaled_prediction(model, X))
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return np.mean(predictions, axis=0)


def bigwig_values(path, chrom, start, end):
    with pybigtools.open(str(path)) as bw:
        return np.nan_to_num(np.asarray(bw.values(chrom, start, end), dtype=float))


def plot_tracks(prediction):
    view_chrom, start, end = parse_interval(VIEW_REGION)
    if view_chrom != chrom:
        raise ValueError("POINT_REGION and VIEW_REGION must use the same chromosome")
    output_start = center - OUT_WINDOW // 2
    output_end = output_start + OUT_WINDOW
    if start < output_start or end > output_end:
        raise ValueError(
            f"VIEW_REGION spans {view_chrom}:{start + 1}-{end}, but predictions cover only "
            f"{view_chrom}:{output_start + 1}-{output_end}."
        )
    left, right = start - output_start, end - output_start
    predicted_plus = prediction[0, left:right]
    predicted_minus = -prediction[1, left:right]
    observed_plus = bigwig_values(resources["observed"]["plus"], chrom, start, end)
    observed_minus = -np.abs(bigwig_values(resources["observed"]["minus"], chrom, start, end))
    x = np.arange(start, end)
    lengths = {len(x), len(predicted_plus), len(predicted_minus), len(observed_plus), len(observed_minus)}
    if len(lengths) != 1:
        raise ValueError(f"Track lengths do not agree: {lengths}")
    if REVERSE_COMPLEMENT:
        observed_plus, observed_minus = reverse_complement_tracks(observed_plus, observed_minus)
        predicted_plus, predicted_minus = reverse_complement_tracks(predicted_plus, predicted_minus)
        x = x[::-1]
    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.plot(x, observed_plus, color="#C44E52", label="observed plus")
    ax.plot(x, observed_minus, color="#4C72B0", label="observed minus")
    ax.plot(x, predicted_plus, color="#C44E52", linestyle="--", label="predicted plus")
    ax.plot(x, predicted_minus, color="#4C72B0", linestyle="--", label="predicted minus")
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_xlim(x[0], x[-1])
    ax.set_title(f"{EXP_ID} {VIEW_REGION}")
    ax.set_ylabel("PRO-cap signal")
    ax.legend(frameon=False, ncol=2)
    return fig, ax

In [ ]:
prediction = ensemble_prediction(resources, X)
plot_tracks(prediction)
plt.show()

## Frequency-reference DeepLIFT

The reference is a single soft sequence with the observed A/C/G/T frequencies from the 2114 bp genomic input repeated at every position. This matches the default BPNet attribution workflow and avoids stochastic shuffled-reference baselines.

In [ ]:
def deeplift_attributions(resources, X):
    references = nucleotide_frequency_references(X)
    width = logo_offsets[1] - logo_offsets[0]
    attributions = {"profile": [], "count": []}
    for fold, path in enumerate(resources["model_paths"][:N_FOLDS]):
        print(f"DeepLIFT fold {fold + 1}/{N_FOLDS}: {path.name}")
        model = torch.load(path, map_location="cpu", weights_only=False).eval()
        wrappers = {"profile": ProfileWrapper(model), "count": CountWrapper(model)}
        for head, wrapper in wrappers.items():
            attr = deep_lift_shap(
                model=wrapper,
                X=X,
                references=references,
                n_shuffles=1,
                batch_size=BATCH_SIZE,
                hypothetical=True,
                warning_threshold=0.01,
                device=DEVICE,
            )
            observed = as_numpy(attr * X)[0, :, logo_offsets[0] : logo_offsets[1]]
            attributions[head].append(observed)
        del model, wrappers
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return {head: np.mean(values, axis=0) for head, values in attributions.items()}


def oriented_logo_matrix(matrix):
    return reverse_complement_matrix(matrix) if REVERSE_COMPLEMENT else matrix


def logo_ticks(ax):
    width = logo_end - logo_start
    positions = np.linspace(0, width - 1, 5, dtype=int)
    labels = [logo_end - position if REVERSE_COMPLEMENT else logo_start + position + 1 for position in positions]
    ax.set_xticks(positions)
    ax.set_xticklabels([f"{value:,}" for value in labels])


def plot_deeplift_logos(attributions):
    fig, axes = plt.subplots(2, 1, figsize=(14, 5.5), sharex=True)
    for ax, head in zip(axes, ["profile", "count"]):
        matrix = oriented_logo_matrix(attributions[head])
        plot_logo(torch.tensor(matrix, dtype=torch.float32), ax=ax)
        ax.set_title(f"{head} DeepLIFT/SHAP, frequency reference")
        logo_ticks(ax)
    fig.suptitle(f"{EXP_ID} {LOGO_REGION}")
    fig.tight_layout()
    return fig, axes

In [ ]:
attributions = deeplift_attributions(resources, X)
plot_deeplift_logos(attributions)
plt.show()

In [ ]:
# Optional: save the current figures and arrays for later inspection.
OUTPUT_DIR = Path("plots/bpnet/locus_viewer") / EXP_ID / POINT_REGION.replace(":", "_").replace(",", "")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plot_tracks(prediction)[0].savefig(OUTPUT_DIR / "observed_predicted_tracks.pdf", bbox_inches="tight")
plot_deeplift_logos(attributions)[0].savefig(OUTPUT_DIR / "frequency_reference_deeplift_logos.pdf", bbox_inches="tight")
np.savez_compressed(
    OUTPUT_DIR / "locus_viewer_arrays.npz",
    prediction=prediction,
    profile_deeplift=attributions["profile"],
    count_deeplift=attributions["count"],
    point_region=np.asarray(POINT_REGION),
    view_region=np.asarray(VIEW_REGION),
    logo_region=np.asarray(LOGO_REGION),
)
plt.close("all")
print(f"Saved {OUTPUT_DIR}")